In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import librosa
import gradio as gr

In [ ]:
# Load model and processor
# model_name = "openai/whisper-base"
model_name = "openai/whisper-large-v3"

model = WhisperForConditionalGeneration.from_pretrained(model_name)
processor = WhisperProcessor.from_pretrained(model_name)

# Set to CPU (or cuda if available)
device = "cpu"
model.to(device)

In [ ]:
def transcribe_audio(audio_path):
    """Transcribe audio file of any length using Whisper's native long-form transcription"""
    if audio_path is None:
        return "No audio recorded"

    # Load and resample audio to 16kHz (Whisper's expected sample rate)
    audio, sr = librosa.load(audio_path, sr=16000)

    # Process audio
    input_features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    # Generate transcription with Finnish language setting
    # Whisper handles long audio automatically through its own chunking
    predicted_ids = model.generate(
        input_features,
        language="fi",
        task="transcribe"
    )

    # Decode the transcription
    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return transcription

# Create Gradio interface
demo = gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
    outputs=gr.Textbox(lines=10),
    title="Speech to Text with Whisper (Long Audio Support)",
    description="Record your voice or upload an audio file of any length to get it transcribed in Finnish"
)

In [ ]:
demo.launch(share=True)